# Курсовая: торговая стратегия на основе ML

In [ ]:
!pip install git+https://github.com/quantiacs/toolbox.git 2>/dev/null
!pip install xgboost optuna colorednoise pmdarima 2>/dev/null

import os
os.environ['API_KEY'] = ''
os.environ['NONINTERACT'] = 'True'

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import sys
import importlib


script_dir = ''
if script_dir not in sys.path:
    sys.path.append(script_dir)

import data_loader
importlib.reload(data_loader)

from data_loader import (
    load_spx_data,
    compute_paper_features,
    build_monthly_dataset,
    extract_report_dates,
    MARKET_FIELDS,
    FUNDAMENTAL_FEATURES,
    TS_FEATURES,
)

BASE = Path('')

market_da, fund_da = load_spx_data(BASE)

print(f"Market: {dict(market_da.sizes)}")
print(f"Fund:   {dict(fund_da.sizes)}")
print(f"Fields market: {list(market_da.coords['field'].values)}")
print(f"Time: {str(market_da.time.values[0])[:10]} → {str(market_da.time.values[-1])[:10]}")
print(f"Assets: {market_da.sizes['asset']}")

## 1. EDA: структура данных SPX

In [ ]:
fund_fields = [f for f in fund_da.coords['field'].values.astype(str) if f not in MARKET_FIELDS]
fund_clean = fund_da.sel(field=fund_fields)

fill_ratio = 1 - fund_clean.isnull().mean(dim=['time', 'asset'])
fill_ratio = fill_ratio.to_pandas().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
fill_ratio.plot.barh(ax=ax)
ax.set_xlabel('Доля заполненных значений')
ax.set_title('Заполненность фундаменталов (SPX)')
plt.tight_layout()
plt.show()

roe_da = fund_da.sel(field='roe')
roe_any = roe_da.notnull().any(dim='time')
n_with_roe = int(roe_any.sum())
print(f'\nАктивов с ROE: {n_with_roe} / {len(roe_any)}')

In [ ]:
assets_list = list(market_da.coords['asset'].values.astype(str))
sample_asset = assets_list[0]
sample = fund_clean.sel(asset=sample_asset).to_pandas().sort_index()
rd = extract_report_dates(sample)
print(f'{sample_asset}: {len(rd)} отчётных дат, '
      f'gap медиана = {pd.Series(rd).diff().dt.days.median():.0f} дней')
print('Первые 5:', rd[:5].tolist())

## 2. Feature Engineering


In [ ]:
features_da = compute_paper_features(market_da, fund_da)

feat_names = list(features_da.coords['field'].values)
print(f'\nВсего фич: {len(feat_names)}')
print(feat_names)

feat_fill = (1 - features_da.isnull().mean(dim=['time', 'asset'])).to_pandas().sort_values(ascending=False)
print('\nЗаполненность фич:')
print(feat_fill)

## 3. Построение ежемесячного датасета

In [ ]:
# HORIZON_DAYS = 30

# df, stats_df = build_monthly_dataset(
#     market_da, features_da=features_da, fund_da=fund_da,
#     horizon_days=HORIZON_DAYS, use_is_liquid=True,
#     require_all_features=False,  # Только строки со ВСЕМИ фичами
# )
# print(f'\nДатасет: {df.shape}')
# print(f'Период: {df["date"].min()} → {df["date"].max()}')
# print(f'Активов: {df["asset"].nunique()}')

# import matplotlib.pyplot as plt
# fig, ax = plt.subplots(figsize=(14, 4))
# ax.bar(stats_df['date'], stats_df['n_assets'], width=25, alpha=0.7)
# ax.axhline(stats_df['n_assets'].mean(), color='red', linestyle='--', 
#            label=f'Mean: {stats_df["n_assets"].mean():.0f}')
# ax.set_xlabel('Дата')
# ax.set_ylabel('Количество активов')
# ax.set_title('Количество активов с полными фичами по месяцам')
# ax.legend()
# plt.tight_layout()
# plt.show()

# df.head()

In [ ]:
# df.to_csv('dataset_monthly_excess.csv', index=False)
# stats_df.to_csv('dataset_monthly_stats.csv', index=False)
# print(f'Датасет сохранён: dataset_monthly_excess.csv ({df.shape})')
# print(f'Статистика сохранена: dataset_monthly_stats.csv ({stats_df.shape})')

In [ ]:
HORIZON_DAYS = 30
df = pd.read_csv('/kaggle/input/datasets/miival/dataset-monthly-excess/dataset_monthly_excess.csv',
                 parse_dates=['date'])
print(f'Датасет загружен: {df.shape}')
print(f'Период: {df["date"].min()} → {df["date"].max()}')
print(f'Активов: {df["asset"].nunique()}, месяцев: {df["date"].nunique()}')
print(f'Активов: {df["asset"].nunique()}')

In [ ]:
# EXTRA_COLS = {'mom9m', 'quarter', 'is_january', 'vol_regime',
#               'str', 'high52w', 'skew', 'abnormal_turn'}

EXTRA_COLS = {'mom9m'}

feature_cols = [c for c in df.columns if c not in {'asset', 'date', 'target'} | EXTRA_COLS]
print(f'Фичей для моделей: {len(feature_cols)}')
print(f'Фичи: {feature_cols}')
extra_present = [c for c in EXTRA_COLS if c in df.columns]
if extra_present:
    print(f'Доп. фичи (не в model features): {extra_present}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['target'].hist(bins=100, ax=axes[0])
axes[0].set_title(f'Excess return ({HORIZON_DAYS}д)')
axes[0].axvline(0, color='r', ls='--')

df.groupby('date')['target'].mean().plot(ax=axes[1])
axes[1].set_title('Средний excess return по месяцам')
axes[1].axhline(0, color='r', ls='--')

df.groupby('date')['asset'].count().plot(ax=axes[2])
axes[2].set_title('Акций в датасете по месяцам')

plt.tight_layout()
plt.show()

# NaN stats
nan_pct = df[feature_cols].isna().mean().sort_values(ascending=False)
print('\nДоля NaN по фичам:')
print(nan_pct[nan_pct > 0])

## 4. Optuna search + обучение моделей

In [ ]:
!pip install xgboost optuna 2>/dev/null

In [ ]:
## import sys
from pathlib import Path

script_dir = ''
if script_dir not in sys.path:
    sys.path.append(script_dir)

from strategy import (
    LSTMReturnModel, LSTMMLPReturnModel, LSTMXGBReturnModel,
    LSTMReturnsOnlyModel, GRUReturnsOnlyModel, LinearReturnModel,
    GRUReturnModel, TransformerReturnModel,
    RFReturnModel, XGBReturnModel,
    train_val_test_split, compute_ic, compute_icir, optuna_search,
    cross_sectional_normalize, normalize_target_monthly,
    DEFAULT_SEARCH_SPACES,
)

# =============================================================================

ENABLED_MODELS = [
    'linreg',         
    'rf',
    'xgb',
    'lstm',           
    # 'lstm_mlp',     
    # 'lstm_xgb',     
    'lstm_ret',       
    'lstm_ts',        
    'gru',            
    'gru_ret',        
    'gru_ts',         
    'transformer',    
    'transformer_ts', 
]


MODELS_TO_LOAD = ['linreg', 'xgb', 'rf', 'lstm', 'lstm_ret', 'lstm_ts', 'gru', 'gru_ret', 'gru_ts', 'transformer', 'transformer_ts']

# "mse" "ic" "ranking" "combined"
LOSS_FUNCTION = "mse"

OPTUNA_METRIC = "ic"

N_TRIALS = 60

USE_CROSS_VALIDATION = False  
CV_FOLDS = 3
CV_GAP_MONTHS = 1 


USE_CROSS_SECTIONAL_NORM = False 


VAL_START = "2018-01-01"
VAL2_START = "2021-01-01"  
TEST_START = "2021-01-01"


SEARCH_SPACES = {
    "xgb": {
        "n_estimators": ("int_step", 100, 1000, 100),
        "max_depth": ("int", 3, 8),
        "learning_rate": ("float_log", 0.01, 0.3),
        "subsample": ("float", 0.5, 1.0),
        "colsample_bytree": ("float", 0.5, 1.0),
    },
    "rf": {
        "n_estimators": ("int_step", 100, 500, 50),
        "max_depth": ("int", 5, 20),
        "min_samples_leaf": ("int", 10, 50),
    },
    "lstm_ret": {
        "lookback": ("categorical", [3, 6, 9, 12, 18]),
        "hidden_size": ("categorical", [16, 32, 64]),
        "num_layers": ("categorical", [1, 2, 3]),
        "dropout": ("float", 0.1, 0.4),
        "lr": ("float_log", 1e-4, 1e-2),
        "epochs": ("int_step", 40, 100, 10),
        "patience": ("fixed", 15),
        "batch_size": ("categorical", [256, 512]),
    },
    "lstm_xgb": {
        "lookback": ("categorical", [3, 6, 9, 12, 18]),
        "lstm_hidden": ("categorical", [8, 16, 32]),
        "lstm_layers": ("categorical", [1, 2]),
        "lstm_dropout": ("float", 0.1, 0.4),
        "lstm_lr": ("float_log", 1e-4, 1e-2),
        "lstm_epochs": ("int_step", 20, 60, 10),
        "lstm_batch_size": ("categorical", [256, 512]),
        "lstm_patience": ("fixed", 10),
        "xgb_n_estimators": ("int_step", 100, 500, 100),
        "xgb_max_depth": ("int", 3, 8),
        "xgb_learning_rate": ("float_log", 0.01, 0.2),
        "xgb_subsample": ("float", 0.6, 1.0),
    },
    "lstm": {
        "lookback": ("categorical", [3, 6, 9, 12, 18]),
        "hidden_size": ("categorical", [32, 64, 128]),
        "num_layers": ("int", 1, 3),
        "dropout": ("float", 0.1, 0.5),
        "lr": ("float_log", 1e-4, 1e-2),
        "epochs": ("int_step", 40, 100, 10),
        "patience": ("fixed", 15),
        "batch_size": ("categorical", [256, 512, 1024]),
    },
    "lstm_mlp": {
        "lookback": ("categorical", [3, 6, 9, 12, 18]),
        "lstm_hidden": ("fixed", 30),
        "lstm_layers": ("categorical", [1, 2]),
        "mlp_hidden": ("categorical", [64, 128, 256]),
        "mlp_depth": ("categorical", [2, 3, 4]),
        "dropout": ("float", 0.1, 0.5),
        "lr": ("float_log", 1e-4, 1e-2),
        "epochs": ("int_step", 40, 100, 10),
        "patience": ("fixed", 15),
        "batch_size": ("categorical", [256, 512, 1024]),
    },
    "gru": {
        "lookback": ("categorical", [3, 6, 9, 12, 18]),
        "hidden_size": ("categorical", [64, 128, 256]),
        "num_layers": ("int", 1, 3),
        "dropout": ("float", 0.1, 0.5),
        "lr": ("float_log", 1e-4, 1e-2),
        "epochs": ("int_step", 40, 100, 10),
        "patience": ("fixed", 15),
        "batch_size": ("categorical", [256, 512, 1024]),
    },
    "transformer": {
        "lookback": ("categorical", [3, 6, 9, 12, 18]),
        "d_model": ("categorical", [32, 64, 128]),
        "nhead": ("categorical", [2, 4, 8]),
        "num_layers": ("int", 1, 3),
        "dim_feedforward": ("categorical", [64, 128, 256]),
        "dropout": ("float", 0.1, 0.5),
        "lr": ("float_log", 1e-4, 1e-2),
        "epochs": ("int_step", 40, 100, 10),
        "patience": ("fixed", 15),
        "batch_size": ("categorical", [256, 512, 1024]),
    },
}

# =============================================================================
MODELS_TO_TRAIN = [m for m in ENABLED_MODELS if m not in (MODELS_TO_LOAD or [])]

print(f'Enabled models: {ENABLED_MODELS}')
print(f'Models to train: {MODELS_TO_TRAIN}')
print(f'Models to load: {MODELS_TO_LOAD or "none"}')
print(f'Loss function: {LOSS_FUNCTION}')
print(f'Optuna metric: {OPTUNA_METRIC}')
print(f'N trials: {N_TRIALS}')
print(f'Cross-validation: {USE_CROSS_VALIDATION} (folds={CV_FOLDS}, gap={CV_GAP_MONTHS})')
print(f'Cross-sectional normalization: {USE_CROSS_SECTIONAL_NORM}')
print(f'Split: val_start={VAL_START}, test_start={TEST_START}')
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

feature_cols = [c for c in df.columns if c not in {'asset', 'date', 'target'} | EXTRA_COLS]
train_df, val_df, test_df = train_val_test_split(df, val_start=VAL_START, test_start=TEST_START)
print(f'Фичей: {len(feature_cols)}')

In [ ]:

ts_feature_cols = [c for c in feature_cols if c in TS_FEATURES]

if USE_CROSS_SECTIONAL_NORM:
    print('Applying cross-sectional normalization...')
    train_df = cross_sectional_normalize(train_df, feature_cols)
    val_df = cross_sectional_normalize(val_df, feature_cols)
    test_df = cross_sectional_normalize(test_df, feature_cols)

# train_df = normalize_target_monthly(train_df)
# val_df = normalize_target_monthly(val_df)

val_optuna_df = val_df[val_df['date'] < VAL2_START].copy()
val2_df = val_df[val_df['date'] >= VAL2_START].copy()
print(f'val_optuna (Optuna): {len(val_optuna_df)}, val2 (подбор top-N): {len(val2_df)}')


best_params = {}
best_ics = {}

optuna_kwargs = dict(
    metric=OPTUNA_METRIC,
    loss_fn=LOSS_FUNCTION,
    use_cv=USE_CROSS_VALIDATION,
    search_spaces=SEARCH_SPACES,
    cv_folds=CV_FOLDS,
    cv_gap_months=CV_GAP_MONTHS,
)

if 'linreg' in MODELS_TO_TRAIN:
    print('=== Linear Regression (Ridge) ===')
    best_params['linreg'], best_ics['linreg'] = optuna_search(
        'linreg', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'rf' in MODELS_TO_TRAIN:
    print('\n=== RF ===')
    best_params['rf'], best_ics['rf'] = optuna_search(
        'rf', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'xgb' in MODELS_TO_TRAIN:
    print('\n=== XGBoost ===')
    best_params['xgb'], best_ics['xgb'] = optuna_search(
        'xgb', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'lstm' in MODELS_TO_TRAIN:
    print('\n=== LSTM (all features) ===')
    best_params['lstm'], best_ics['lstm'] = optuna_search(
        'lstm', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'lstm_mlp' in MODELS_TO_TRAIN:
    print(f'\n=== LSTM+MLP (LSTM on returns, MLP on {len(feature_cols)} features) ===')
    best_params['lstm_mlp'], best_ics['lstm_mlp'] = optuna_search(
        'lstm_mlp', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'lstm_ret' in MODELS_TO_TRAIN:
    print(f'\n=== LSTM-Ret (LSTM only on z-scored returns) ===')
    best_params['lstm_ret'], best_ics['lstm_ret'] = optuna_search(
        'lstm_ret', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'lstm_xgb' in MODELS_TO_TRAIN:
    print(f'\n=== LSTM+XGB (own LSTM + XGBoost on {len(feature_cols)} features) ===')
    best_params['lstm_xgb'], best_ics['lstm_xgb'] = optuna_search(
        'lstm_xgb', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'lstm_ts' in MODELS_TO_TRAIN:
    print(f'\n=== LSTM-TS ({len(ts_feature_cols)} time-series features) ===')
    print(f'TS features: {ts_feature_cols}')
    best_params['lstm_ts'], best_ics['lstm_ts'] = optuna_search(
        'lstm_ts', train_df, val_optuna_df, ts_feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'gru_ret' in MODELS_TO_TRAIN:
    print(f'\n=== GRU-Ret (GRU only on z-scored returns) ===')
    best_params['gru_ret'], best_ics['gru_ret'] = optuna_search(
        'gru_ret', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'gru' in MODELS_TO_TRAIN:
    print(f'\n=== GRU ({len(feature_cols)} all features) ===')
    best_params['gru'], best_ics['gru'] = optuna_search(
        'gru', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'gru_ts' in MODELS_TO_TRAIN:
    print(f'\n=== GRU-TS ({len(ts_feature_cols)} time-series features) ===')
    best_params['gru_ts'], best_ics['gru_ts'] = optuna_search(
        'gru_ts', train_df, val_optuna_df, ts_feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'transformer' in MODELS_TO_TRAIN:
    print(f'\n=== Transformer ({len(feature_cols)} all features) ===')
    best_params['transformer'], best_ics['transformer'] = optuna_search(
        'transformer', train_df, val_optuna_df, feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

if 'transformer_ts' in MODELS_TO_TRAIN:
    print(f'\n=== Transformer-TS ({len(ts_feature_cols)} time-series features) ===')
    best_params['transformer_ts'], best_ics['transformer_ts'] = optuna_search(
        'transformer_ts', train_df, val_optuna_df, ts_feature_cols, n_trials=N_TRIALS, **optuna_kwargs)

metric_name = OPTUNA_METRIC.upper()
print(f'\n=== Лучшие {metric_name} на валидации ===')
for m in MODELS_TO_TRAIN:
    if m in best_ics:
        print(f'{m.upper():12s}: {best_ics[m]:.4f}  params: {best_params[m]}')

In [ ]:
scaler = StandardScaler()
train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

_clean = lambda x: x.replace([np.inf, -np.inf], np.nan).fillna(0)
train_scaled[feature_cols] = scaler.fit_transform(_clean(train_scaled[feature_cols]))
val_scaled[feature_cols] = scaler.transform(_clean(val_scaled[feature_cols]))
test_scaled[feature_cols] = scaler.transform(_clean(test_scaled[feature_cols]))

scaler_ts = StandardScaler()
train_ts = train_df.copy()
val_ts = val_df.copy()
test_ts = test_df.copy()
train_ts[ts_feature_cols] = scaler_ts.fit_transform(_clean(train_ts[ts_feature_cols]))
val_ts[ts_feature_cols] = scaler_ts.transform(_clean(val_ts[ts_feature_cols]))
test_ts[ts_feature_cols] = scaler_ts.transform(_clean(test_ts[ts_feature_cols]))

full_train = pd.concat([train_scaled, val_scaled]).reset_index(drop=True)
split_80 = int(len(full_train) * 0.8)
ft_train = full_train.iloc[:split_80].copy()
ft_es = full_train.iloc[split_80:].copy()

ts_full = pd.concat([train_ts, val_ts]).reset_index(drop=True)
split_ts = int(len(ts_full) * 0.8)
ts_train = ts_full.iloc[:split_ts].copy()
ts_es = ts_full.iloc[split_ts:].copy()

trained_models = {}

if 'linreg' in MODELS_TO_TRAIN:
    print('=== Training LinearReg (Ridge) ===')
    linreg_model = LinearReturnModel(**best_params['linreg'])
    linreg_model.fit(ft_train, ft_es, feature_cols)
    trained_models['linreg'] = linreg_model

if 'rf' in MODELS_TO_TRAIN:
    print('=== Training RF ===')
    rf_model = RFReturnModel(**best_params['rf'])
    rf_model.fit(ft_train, ft_es, feature_cols)
    trained_models['rf'] = rf_model

if 'xgb' in MODELS_TO_TRAIN:
    print('=== Training XGBoost ===')
    xgb_model = XGBReturnModel(**best_params['xgb'])
    xgb_model.fit(ft_train, ft_es, feature_cols)
    trained_models['xgb'] = xgb_model

if 'lstm' in MODELS_TO_TRAIN:
    print('=== Training LSTM ===')
    lstm_model = LSTMReturnModel(**best_params['lstm'])
    lstm_model.fit(ft_train, ft_es, feature_cols)
    trained_models['lstm'] = lstm_model

if 'lstm_mlp' in MODELS_TO_TRAIN:
    print('=== Training LSTM+MLP ===')
    lstm_mlp_model = LSTMMLPReturnModel(**best_params['lstm_mlp'])
    lstm_mlp_model.fit(ft_train, ft_es, feature_cols)
    trained_models['lstm_mlp'] = lstm_mlp_model

if 'lstm_ret' in MODELS_TO_TRAIN:
    print('=== Training LSTM-Ret ===')
    lstm_ret_model = LSTMReturnsOnlyModel(**best_params['lstm_ret'])
    lstm_ret_model.fit(ft_train, ft_es, feature_cols)
    trained_models['lstm_ret'] = lstm_ret_model

if 'lstm_xgb' in MODELS_TO_TRAIN:
    print('=== Training LSTM+XGB ===')
    lstm_xgb_model = LSTMXGBReturnModel(**best_params['lstm_xgb'])
    lstm_xgb_model.fit(ft_train, ft_es, feature_cols)
    trained_models['lstm_xgb'] = lstm_xgb_model

if 'lstm_ts' in MODELS_TO_TRAIN:
    print('=== Training LSTM-TS ===')
    lstm_ts_model = LSTMReturnModel(**best_params['lstm_ts'])
    lstm_ts_model.fit(ts_train, ts_es, ts_feature_cols)
    trained_models['lstm_ts'] = lstm_ts_model

if 'gru_ret' in MODELS_TO_TRAIN:
    print('=== Training GRU-Ret ===')
    gru_ret_model = GRUReturnsOnlyModel(**best_params['gru_ret'])
    gru_ret_model.fit(ft_train, ft_es, feature_cols)
    trained_models['gru_ret'] = gru_ret_model

if 'gru' in MODELS_TO_TRAIN:
    print('=== Training GRU ===')
    gru_model = GRUReturnModel(**best_params['gru'])
    gru_model.fit(ft_train, ft_es, feature_cols)
    trained_models['gru'] = gru_model

if 'gru_ts' in MODELS_TO_TRAIN:
    print('=== Training GRU-TS ===')
    gru_ts_model = GRUReturnModel(**best_params['gru_ts'])
    gru_ts_model.fit(ts_train, ts_es, ts_feature_cols)
    trained_models['gru_ts'] = gru_ts_model

if 'transformer' in MODELS_TO_TRAIN:
    print('=== Training Transformer ===')
    tf_model = TransformerReturnModel(**best_params['transformer'])
    tf_model.fit(ft_train, ft_es, feature_cols)
    trained_models['transformer'] = tf_model

if 'transformer_ts' in MODELS_TO_TRAIN:
    print('=== Training Transformer-TS ===')
    tf_ts_model = TransformerReturnModel(**best_params['transformer_ts'])
    tf_ts_model.fit(ts_train, ts_es, ts_feature_cols)
    trained_models['transformer_ts'] = tf_ts_model


print(f'Trained {len(trained_models)} models: {list(trained_models.keys())}')


In [ ]:
import pickle
import torch
from pathlib import Path
from strategy import (
    ReturnLSTM, ReturnLSTMMLP, ReturnGRU, ReturnTransformer,
    ReturnLSTMReturnsOnly, ReturnGRUReturnsOnly,
)

LOAD_DIR = Path('')

def load_models(models_to_load, path=LOAD_DIR):
    loaded_bp = {}
    bp_path = path / 'best_params.pkl'
    if bp_path.exists():
        with open(bp_path, 'rb') as f:
            loaded_bp = pickle.load(f)

    loaded = {}
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_MODELS = {'lstm', 'lstm_ts', 'lstm_mlp', 'gru', 'gru_ts', 'transformer', 'transformer_ts'}

    for name in models_to_load:
        torch_path = path / f'{name}_torch.pt'
        sklearn_path = path / f'{name}_sklearn.pkl'
        params = loaded_bp.get(name, {})

        if torch_path.exists() and name in TORCH_MODELS:
            checkpoint = torch.load(torch_path, map_location=device, weights_only=False)
            params = checkpoint['params']
            if name in ('lstm', 'lstm_ts'):
                model = LSTMReturnModel(**params)
            elif name == 'lstm_mlp':
                model = LSTMMLPReturnModel(**params)
            elif name in ('gru', 'gru_ts'):
                model = GRUReturnModel(**params)
            elif name in ('transformer', 'transformer_ts'):
                model = TransformerReturnModel(**params)
            else:
                continue

            n_feat = len(feature_cols) if name not in ['lstm_ts', 'gru_ts', 'transformer_ts'] else len(ts_feature_cols)
            if name == 'lstm_mlp':
                model.model = ReturnLSTMMLP(n_feat, model.lstm_hidden,
                    model.lstm_layers, model.mlp_hidden,
                    model.mlp_depth, model.dropout).to(device)
            elif name in ('lstm', 'lstm_ts'):
                model.model = ReturnLSTM(n_feat, model.hidden_size,
                    model.num_layers, model.dropout).to(device)
            elif name in ('gru', 'gru_ts'):
                model.model = ReturnGRU(n_feat, model.hidden_size,
                    model.num_layers, model.dropout).to(device)
            elif name in ('transformer', 'transformer_ts'):
                model.model = ReturnTransformer(n_feat, model.d_model, model.nhead,
                    model.num_layers, model.dim_feedforward,
                    model.dropout).to(device)

            model.model.load_state_dict(checkpoint['state_dict'])
            model.device = device
            loaded[name] = model
            print(f'Loaded: {name} (PyTorch)')

        elif sklearn_path.exists():
            if name == 'linreg':
                model = LinearReturnModel(**params)
            elif name == 'rf':
                model = RFReturnModel(**params)
            elif name == 'xgb':
                model = XGBReturnModel(**params)
            elif name == 'lstm_ret':
                model = LSTMReturnsOnlyModel(**params)
            elif name == 'gru_ret':
                model = GRUReturnsOnlyModel(**params)
            elif name == 'lstm_xgb':
                model = LSTMXGBReturnModel(**params)
            else:
                print(f'Skipped (unknown): {name}')
                continue

            with open(sklearn_path, 'rb') as f:
                model.model = pickle.load(f)
            if hasattr(model, 'device'):
                model.device = device
            loaded[name] = model
            print(f'Loaded: {name} (pickle)')
        else:
            print(f'Not found: {name}')

    print(f'\nLoaded {len(loaded)}/{len(models_to_load)} models from {path}/')
    return loaded, loaded_bp

if MODELS_TO_LOAD:
    _loaded, _loaded_bp = load_models(MODELS_TO_LOAD, path=LOAD_DIR)
    trained_models.update(_loaded)
    best_params.update({k: v for k, v in _loaded_bp.items() if k in MODELS_TO_LOAD})
    print(f'\nUnified trained_models: {list(trained_models.keys())}')
else:
    print('MODELS_TO_LOAD пуст')

print(f'Total models ready: {len(trained_models)}')


In [ ]:
TS_MODELS_SET = {'lstm_ts', 'gru_ts', 'transformer_ts'}
SEQ_MODELS_SET = {'lstm', 'lstm_mlp', 'lstm_ret', 'gru', 'gru_ret', 'lstm_xgb',
                  'lstm_ts', 'gru_ts', 'transformer', 'transformer_ts'}

for m, model in trained_models.items():
    col = f'pred_{m}'
    print(f'Predicting {m}...', end=' ')
    if m in TS_MODELS_SET:
        test_ts[col] = model.predict(test_ts, ts_feature_cols, history_df=ts_full)
    elif m in SEQ_MODELS_SET:
        test_scaled[col] = model.predict(test_scaled, feature_cols, history_df=full_train)
    else:
        test_scaled[col] = model.predict(test_scaled, feature_cols)
    print('done')

test_result = test_df.copy()
pred_cols = []
for m in trained_models:
    col = f'pred_{m}'
    if m in TS_MODELS_SET:
        if col in test_ts.columns:
            test_result[col] = test_ts[col].values
            pred_cols.append(col)
    else:
        if col in test_scaled.columns:
            test_result[col] = test_scaled[col].values
            pred_cols.append(col)

print(f'\nTest samples: {len(test_result)}')
for col in pred_cols:
    print(f'{col}: {test_result[col].isna().sum()} NaN preds')


In [ ]:
# === СОХРАНЕНИЕ МОДЕЛЕЙ ===
import pickle
import torch
from pathlib import Path

SAVE_DIR = Path('saved_models')
SAVE_DIR.mkdir(exist_ok=True)

def save_models(trained_models, best_params, scaler, scaler_ts, path=SAVE_DIR):
    """Сохраняет все обученные модели и параметры."""
    with open(path / 'best_params.pkl', 'wb') as f:
        pickle.dump(best_params, f)
    with open(path / 'scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    with open(path / 'scaler_ts.pkl', 'wb') as f:
        pickle.dump(scaler_ts, f)

    pytorch_models = {'lstm', 'lstm_ts', 'lstm_mlp', 'gru', 'gru_ts', 'transformer', 'transformer_ts'}
    for name, model in trained_models.items():
        if model is None or not hasattr(model, 'model') or model.model is None:
            print(f'Skipped (no model): {name}')
            continue
        if name in pytorch_models:
            torch.save({
                'state_dict': model.model.state_dict(),
                'params': {k: v for k, v in model.__dict__.items()
                          if k not in ['model', 'device', 'train_losses_', 'val_losses_']}
            }, path / f'{name}_torch.pt')
            print(f'Saved: {name} (PyTorch)')
        else:
            with open(path / f'{name}_sklearn.pkl', 'wb') as f:
                pickle.dump(model.model, f)
            print(f'Saved: {name} (sklearn/pickle)')

    print(f'\nAll models saved to {path}/')

save_models(trained_models, best_params, scaler, scaler_ts)


In [ ]:
from scipy.stats import spearmanr

MODEL_NAME_MAP = {
    'linreg': 'LinReg',
    'rf': 'RF',
    'xgb': 'XGBoost',
    'lstm': 'LSTM',
    'lstm_mlp': 'LSTM+MLP',
    'lstm_xgb': 'LSTM+XGB',
    'lstm_ret': 'LSTM-Ret',
    'lstm_ts': 'LSTM-TS',
    'gru': 'GRU',
    'gru_ret': 'GRU-Ret',
    'gru_ts': 'GRU-TS',
    'transformer': 'Transformer',
    'transformer_ts': 'Transformer-TS',
}
models_info = [(MODEL_NAME_MAP[m], f'pred_{m}') for m in ENABLED_MODELS if m in MODEL_NAME_MAP]
print(f'Models for evaluation: {[m[0] for m in models_info]}')

rows = []
for name, col in models_info:
    t = test_result.dropna(subset=[col, 'target'])
    if len(t) < 10:
        continue
    ic = compute_ic(t.rename(columns={col: 'pred'}))
    pearson = t[['target', col]].corr().iloc[0, 1]
    spear = spearmanr(t['target'], t[col])[0]
    mae = mean_absolute_error(t['target'], t[col])
    rmse = mean_squared_error(t['target'], t[col]) ** 0.5
    rows.append({
        'Model': name, 'IC': f'{ic:.4f}',
        'Pearson': f'{pearson:.4f}', 'Spearman': f'{spear:.4f}',
        'MAE': f'{mae:.4f}', 'RMSE': f'{rmse:.4f}',
        'N': len(t),
    })

quality_df = pd.DataFrame(rows)
print('=== Prediction Quality (Test) ===')
print(quality_df.to_string(index=False))

## 4b. Per-month IC и диагностика предсказаний

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ic_monthly = []
for name, col in models_info:
    t = test_result.dropna(subset=[col, 'target']).copy()
    t['month'] = t['date'].dt.to_period('M')
    for m, grp in t.groupby('month'):
        if len(grp) < 5:
            continue
        ic_val, _ = spearmanr(grp[col], grp['target'])
        ic_monthly.append({'month': m.to_timestamp(), 'model': name, 'IC': ic_val})

ic_df = pd.DataFrame(ic_monthly)
for name in ['RF', 'XGBoost', 'LSTM']:
    sub = ic_df[ic_df['model'] == name]
    axes[0].plot(sub['month'], sub['IC'], label=name, alpha=0.8)
axes[0].axhline(0, color='grey', ls='--')
axes[0].set_title('Monthly IC (test)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i, (name, col) in enumerate(models_info):
    t = test_result.dropna(subset=[col, 'target'])
    axes[1].scatter(t[col], t['target'], alpha=0.05, s=3, label=name)
axes[1].set_xlabel('Predicted excess return')
axes[1].set_ylabel('Actual excess return')
axes[1].set_title('Predicted vs Actual')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if 'rf' in trained_models and trained_models['rf'].model is not None:
    try:
        imp_rf = pd.Series(trained_models['rf'].model.feature_importances_, index=feature_cols)
        imp_rf.sort_values().tail(15).plot.barh(ax=axes[0])
        axes[0].set_title('RF: Top-15 Feature Importance')
    except AttributeError:
        axes[0].set_title('RF: feature_importances_ not available (cuML)')

if 'xgb' in trained_models and trained_models['xgb'].model is not None:
    imp_xgb = pd.Series(trained_models['xgb'].model.feature_importances_, index=feature_cols)
    imp_xgb.sort_values().tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('XGBoost: Top-15 Feature Importance')

plt.tight_layout()
plt.show()

## 5. Бэктест

In [ ]:
TOP_N_GRID = [5, 10, 20, 30, 50, 100, 150, 200]

In [ ]:
from robustness import custom_backtest

best_top_n_test = {}
ALL_MODELS = [MODEL_NAME_MAP[m] for m in ENABLED_MODELS if m in MODEL_NAME_MAP]
if 'pred_momentum' in test_result.columns:
    ALL_MODELS = ALL_MODELS + ['Momentum']

models_pred_map = {MODEL_NAME_MAP[m]: f'pred_{m}' for m in ENABLED_MODELS if m in MODEL_NAME_MAP}
models_pred_map['Momentum'] = 'pred_momentum'

for model_name in ALL_MODELS:
    pred_col = models_pred_map.get(model_name)
    if pred_col is None or pred_col not in test_result.columns:
        best_top_n_test[model_name] = 30
        continue
    best_sr, best_n = -999, 30
    for top_n in TOP_N_GRID:
        bt = custom_backtest(test_result, pred_col, market_da, top_n=top_n)
        sr = bt['metrics'].get('sharpe', -999)
        if sr > best_sr:
            best_sr = sr
            best_n = top_n
    best_top_n_test[model_name] = best_n
    print(f'  {model_name}: top-{best_n} (Sharpe={best_sr:.3f})')

best_top_n_test['EW'] = 0
print(f'\nOracle best top-N on test: {best_top_n_test}')

### Custom Backtest 

In [ ]:
import importlib
import robustness
importlib.reload(robustness)
from robustness import custom_backtest, run_custom_backtest_all, plot_custom_backtest

In [ ]:
# from robustness import custom_backtest, run_custom_backtest_all, plot_custom_backtest
# from typing import Dict, List, Optional, Tuple, Any

custom_equities, custom_metrics = run_custom_backtest_all(
    test_result, market_da, models_info, 
    top_n_grid=TOP_N_GRID, slippage_bps=10.0
)

display_df = custom_metrics.copy()
display_df['Total Return'] = display_df['Total Return'].apply(lambda x: f'{x:.1%}')
display_df['Annual Return'] = display_df['Annual Return'].apply(lambda x: f'{x:.1%}')
display_df['Volatility'] = display_df['Volatility'].apply(lambda x: f'{x:.1%}')
display_df['Sharpe'] = display_df['Sharpe'].apply(lambda x: f'{x:.3f}')
display_df['Sortino'] = display_df['Sortino'].apply(lambda x: f'{x:.3f}')
display_df['Max DD'] = display_df['Max DD'].apply(lambda x: f'{x:.1%}')
display_df[['Strategy', 'Total Return', 'Annual Return', 'Volatility', 'Sharpe', 'Sortino', 'Max DD']]

In [ ]:
plot_custom_backtest(custom_equities, custom_metrics, TOP_N_GRID)

from robustness import MODEL_COLORS

VAL_TOP_N_PLOT = 50

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for model_name in custom_metrics['Model'].unique():
    if model_name == 'EW':
        continue
    key = f'{model_name} top-{VAL_TOP_N_PLOT}'
    if key in custom_equities:
        eq = custom_equities[key]
        row = custom_metrics[custom_metrics['Strategy'] == key]
        sr = row['Sharpe'].iloc[0] if len(row) else 0
        ls = '--' if model_name == 'Momentum' else '-'
        axes[0].plot(eq.index, eq.values,
                     label=f'{key} (SR={sr:.2f})',
                     color=MODEL_COLORS.get(model_name, 'grey'), lw=1.5, ls=ls)

if 'EW (all)' in custom_equities:
    ew_row = custom_metrics[custom_metrics['Model'] == 'EW']
    sr = ew_row['Sharpe'].iloc[0] if len(ew_row) else 0
    axes[0].plot(custom_equities['EW (all)'].index, custom_equities['EW (all)'].values,
                 label=f'EW (all) (SR={sr:.2f})',
                 color='black', lw=2, ls=':')

axes[0].axhline(1.0, color='grey', ls=':', alpha=0.5)
axes[0].set_title(f'Custom Backtest: val top-N = {VAL_TOP_N_PLOT}')
axes[0].set_ylabel('Equity')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

for model_name in custom_metrics['Model'].unique():
    if model_name == 'EW':
        continue
    best_n = best_top_n_test.get(model_name, 30)
    key = f'{model_name} top-{best_n}'
    if key in custom_equities:
        eq = custom_equities[key]
        row = custom_metrics[custom_metrics['Strategy'] == key]
        sr = row['Sharpe'].iloc[0] if len(row) else 0
        ls = '--' if model_name == 'Momentum' else '-'
        axes[1].plot(eq.index, eq.values,
                     label=f'{key} (SR={sr:.2f})',
                     color=MODEL_COLORS.get(model_name, 'grey'), lw=1.5, ls=ls)

if 'EW (all)' in custom_equities:
    ew_row = custom_metrics[custom_metrics['Model'] == 'EW']
    sr = ew_row['Sharpe'].iloc[0] if len(ew_row) else 0
    axes[1].plot(custom_equities['EW (all)'].index, custom_equities['EW (all)'].values,
                 label=f'EW (all) (SR={sr:.2f})',
                 color='black', lw=2, ls=':')

axes[1].axhline(1.0, color='grey', ls=':', alpha=0.5)
axes[1].set_title('Custom Backtest: test-oracle best top-N')
axes[1].set_ylabel('Equity')
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5.5 Robustness

In [ ]:
robustness_dir = ''
if robustness_dir not in sys.path:
    sys.path.append(robustness_dir)

import robustness as _rob
importlib.reload(_rob)

from robustness import (
    run_robustness_grid,
    run_robustness_tables,
    compute_topn_stability,
    custom_backtest,
    plot_robustness_all,
    sortino_from_equity,
    SIGMA_GRID, NOISE_COLORS,
)

In [ ]:
FIXED_TOP_N = 30
VAL_TOP_N = 50

ALL_MODELS = [MODEL_NAME_MAP[m] for m in ENABLED_MODELS if m in MODEL_NAME_MAP]
if 'pred_momentum' in test_result.columns:
    ALL_MODELS = ALL_MODELS + ['Momentum']

best_top_n_val = {m: VAL_TOP_N for m in ALL_MODELS}
best_top_n_val['EW'] = 0
print(f'Best top-N per model (hardcoded val): {best_top_n_val}')

from scipy.stats import spearmanr as _sp

pred_col_map = {MODEL_NAME_MAP[m]: f'pred_{m}' for m in ENABLED_MODELS if m in MODEL_NAME_MAP}
pred_col_map['Momentum'] = 'pred_momentum'

baseline_rows = []
for model_name in ALL_MODELS:
    best_n = best_top_n_val[model_name]
    key = f'{model_name} top-{best_n}'
    model_rows = custom_metrics[custom_metrics['Strategy'] == key]
    if len(model_rows) == 0:
        continue
    m = model_rows.iloc[0]

    pc = pred_col_map.get(model_name, None)
    ic_val = np.nan
    if pc and pc in test_result.columns:
        _t = test_result.dropna(subset=[pc, 'target']).copy()
        _t['month'] = pd.to_datetime(_t['date']).dt.to_period('M')
        _ics = []
        for _, _g in _t.groupby('month'):
            if len(_g) >= 5:
                _ic, _ = _sp(_g[pc], _g['target'])
                if not np.isnan(_ic):
                    _ics.append(_ic)
        ic_val = float(np.mean(_ics)) if _ics else np.nan

    baseline_rows.append({
        'model': model_name,
        'sharpe': m['Sharpe'], 'sortino': m['Sortino'],
        'annual_return': m['Annual Return'], 'max_drawdown': m['Max DD'],
        'ic': ic_val,
    })

ew_rows = custom_metrics[custom_metrics['Model'] == 'EW']
if len(ew_rows) > 0:
    ew = ew_rows.iloc[0]
    baseline_rows.append({
        'model': 'EW',
        'sharpe': ew['Sharpe'], 'sortino': ew['Sortino'],
        'annual_return': ew['Annual Return'], 'max_drawdown': ew['Max DD'],
        'ic': np.nan,
    })

baseline_df = pd.DataFrame(baseline_rows).set_index('model')
print('\n=== Baseline (no noise, custom backtest) ===')
print(baseline_df.round(3))

In [ ]:
full_history_all = pd.concat([train_scaled, val_scaled]).reset_index(drop=True)
full_history_ts = pd.concat([train_ts, val_ts]).reset_index(drop=True)

MODEL_CONFIG = {
    'linreg':      (feature_cols,    False, None,             scaler),
    'rf':          (feature_cols,    False, None,             scaler),
    'xgb':         (feature_cols,    False, None,             scaler),
    'lstm':        (feature_cols,    True,  full_history_all, scaler),
    'lstm_mlp':    (feature_cols,    True,  full_history_all, scaler),
    'lstm_xgb':    (feature_cols,    True,  full_history_all, scaler),
    'lstm_ret':    (feature_cols,    True,  full_history_all, scaler),
    'gru':         (feature_cols,    True,  full_history_all, scaler),
    'gru_ret':     (feature_cols,    True,  full_history_all, scaler),
    'lstm_ts':     (ts_feature_cols, True,  full_history_ts,  scaler_ts),
    'gru_ts':         (ts_feature_cols, True,  full_history_ts,  scaler_ts),
    'transformer':    (feature_cols,    True,  full_history_all, scaler),
    'transformer_ts': (ts_feature_cols, True,  full_history_ts,  scaler_ts),
}

models_dict = {}
for m in ENABLED_MODELS:
    if m in trained_models and m in MODEL_CONFIG:
        cfg = MODEL_CONFIG[m]
        display_name = MODEL_NAME_MAP.get(m, m.upper())
        models_dict[display_name] = (trained_models[m], *cfg)

print(f'Models for robustness: {list(models_dict.keys())}')
print(f'Top-N modes: fixed={FIXED_TOP_N}, val_best, test_best')

raw_history_df = pd.concat([train_df, val_df]).reset_index(drop=True)

model_clean, model_noisy, portfolio_clean, portfolio_noisy = run_robustness_tables(
    models_dict=models_dict,
    market_da=market_da,
    fund_da=fund_da,
    test_df=test_df,
    raw_history_df=raw_history_df,
    scaler=scaler,
    feature_cols=feature_cols,
    compute_features_fn=compute_paper_features,
    custom_backtest_fn=custom_backtest,
    top_n_fixed=FIXED_TOP_N,
    top_n_val=best_top_n_val,
    top_n_test=best_top_n_test,
    sigma_grid=SIGMA_GRID,
    colors=list(NOISE_COLORS.keys()),
    test_start='2019-01-01',
    seed=42,
    clean_test_result=test_result,
    n_stability_values=[10, 30, 50, 100],
    ew_weights_fn=None,
    mom_col='mom9m',
)

print(f'\n=== Model-level (clean target) ===')
print(f'Rows: {len(model_clean)}')
display(model_clean.head(5))

print(f'\n=== Model-level (noisy target) ===')
print(f'Rows: {len(model_noisy)}')
display(model_noisy.head(5))

print(f'\n=== Portfolio (clean market) ===')
print(f'Rows: {len(portfolio_clean)}')
display(portfolio_clean.head(5))

print(f'\n=== Portfolio (noisy market) ===')
print(f'Rows: {len(portfolio_noisy)}')
display(portfolio_noisy.head(5))

In [ ]:
key_sigmas = [0.05, 0.1, 0.3, 0.5, 1.0]
all_models_in_results = portfolio_clean['model'].unique()

for metric in ['sharpe', 'sortino', 'annual_return', 'max_drawdown']:
    summary_rows = []
    df_best = portfolio_clean[portfolio_clean['top_n_mode'] == 'val_best']
    for model_name in all_models_in_results:
        if model_name not in baseline_df.index:
            continue
        if metric not in baseline_df.columns:
            continue
        base_val = baseline_df.loc[model_name, metric]
        row = {'Model': model_name, 'Baseline': f'{base_val:.3f}'}
        sub = df_best[df_best['model'] == model_name]
        for sigma in key_sigmas:
            s_sub = sub[sub['sigma'] == sigma]
            if len(s_sub) == 0:
                row[f'σ={sigma}'] = 'N/A'
                continue
            avg_val = s_sub[metric].mean()
            pct_change = (avg_val - base_val) / abs(base_val) * 100 if abs(base_val) > 1e-10 else 0
            row[f'σ={sigma}'] = f'{avg_val:.3f} ({pct_change:+.1f}%)'
        summary_rows.append(row)
    
    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).set_index('Model')
        print(f'\n=== Robustness Summary: {metric} Degradation (val_best top-N) ===')
        display(summary_df)

In [ ]:
import os, datetime

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")


path_custom = os.path.join(RESULTS_DIR, f"strategies_custom_{timestamp}.csv")
custom_metrics.to_csv(path_custom, index=False)
print(f"[1/5] Strategies (custom)    → {path_custom}  ({len(custom_metrics)} rows)")

path_base = os.path.join(RESULTS_DIR, f"baseline_{timestamp}.csv")
baseline_df.to_csv(path_base)
print(f"[2/5] Baseline (no noise)    → {path_base}  ({len(baseline_df)} rows)")


for label, df in [('model_clean', model_clean), ('model_noisy', model_noisy),
                   ('portfolio_clean', portfolio_clean), ('portfolio_noisy', portfolio_noisy)]:
    p = os.path.join(RESULTS_DIR, f'robustness_{label}_{timestamp}.csv')
    df.to_csv(p, index=False)
    print(f'Robustness {label} → {p}  ({len(df)} rows)')

params_rows = []
for m in ENABLED_MODELS:
    if m in best_params:
        params_rows.append({
            'model': MODEL_NAME_MAP.get(m, m),
            'model_key': m,
            'val_metric': OPTUNA_METRIC,
            'val_score': best_ics.get(m, np.nan),
            'best_top_n': best_top_n_val.get(MODEL_NAME_MAP.get(m, m), np.nan),
            'params': str(best_params[m]),
        })
params_df = pd.DataFrame(params_rows)
path_params = os.path.join(RESULTS_DIR, f"best_params_{timestamp}.csv")
params_df.to_csv(path_params, index=False)
print(f"[5/5] Best params & scores   → {path_params}  ({len(params_df)} rows)")